In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# -----------------------
# 1. Fake tabular dataset
# -----------------------
# Let's pretend each sample has 4 features, and we have 100 samples
X = torch.randn(100, 4)        # shape [100, 4]
y = torch.randint(0, 3, (100,)) # 3-class labels (0,1,2)

dataset = TensorDataset(X, y)

# Dataloader splits into mini-batches
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

In [3]:
# -----------------------
# 2. Encoder + Head
# -----------------------
class Encoder(nn.Module):
    def __init__(self, d_in=4, d_feat=8):
        super().__init__()
        self.fc = nn.Linear(d_in, d_feat)
    def forward(self, x):
        return F.relu(self.fc(x))  # [B, d_feat]

class Head(nn.Module):
    def __init__(self, d_feat=8, n_classes=3):
        super().__init__()
        self.fc = nn.Linear(d_feat, n_classes)
    def forward(self, z):
        return self.fc(z)

encoder = Encoder()
head = Head()

In [4]:
# -----------------------
# 3. One training step
# -----------------------
opt = torch.optim.Adam(list(encoder.parameters()) + list(head.parameters()), lr=1e-2)

for batch_x, batch_y in dataloader:
    # Step A: dataloader gives us raw features + labels
    print("Batch shape:", batch_x.shape, batch_y.shape)

    # Step B: encoder transforms features → embeddings
    z = encoder(batch_x)
    print("Encoded features:", z.shape)

    # Step C: head maps embeddings → class logits
    logits = head(z)
    print("Logits:", logits.shape)

    # Step D: compute loss and update
    loss = F.cross_entropy(logits, batch_y)
    opt.zero_grad()
    loss.backward()
    opt.step()

    print("Loss:", loss.item())
    break  # only run one batch for demo


Batch shape: torch.Size([8, 4]) torch.Size([8])
Encoded features: torch.Size([8, 8])
Logits: torch.Size([8, 3])
Loss: 1.1004198789596558


In [5]:
a = torch.tensor([[1, 2, 3], [2, 3, 4]])   # shape [3]
b = torch.tensor([[4, 5, 6], [5, 6, 7]])   # shape [3]

c = torch.cat([a, b], dim=0)
print(c)
print(c.shape)


tensor([[1, 2, 3],
        [2, 3, 4],
        [4, 5, 6],
        [5, 6, 7]])
torch.Size([4, 3])


In [6]:
c = torch.cat([a, b], dim=1)
print(c)
print(c.shape)

tensor([[1, 2, 3, 4, 5, 6],
        [2, 3, 4, 5, 6, 7]])
torch.Size([2, 6])


In [7]:
import pandas as pd
import numpy as np

In [8]:
df_raw = pd.read_csv('health_care_diabetes.csv')

In [13]:
df_raw.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [19]:
a_array = df_raw.iloc[0].to_numpy()
a_num = int(a_array[0])

In [22]:
TESTVAR = a_num
print (TESTVAR)

6
